# Processament dels resultats d'AlphaFold3 - IL-7Rα

Aquest notebook recol·lecta i organitza els resultats de les prediccions estructurals d'AlphaFold3 per als 95 complexos del dataset IL-7Rα.

La funció collectAlphaFold3Results de la classe sequenceModels recorre la carpeta af3_jobs_il7ra, llegeix els fitxers de sortida de cada complex i extreu les mètriques de confiança: ranking_score, summary_iptm, summary_ptm, summary_ranking_score, summary_fraction_disordered, summary_has_clash, summary_chain_iptm, summary_chain_pair_iptm, summary_chain_pair_pae_min i summary_chain_ptm. Per a cada complex es generen 5 estructures (samples 0–4); la funció retorna tant el dataframe amb tots els resultats com el subset amb el millor model per complex (seleccionat per ranking_score màxim). D'alguns complexos (Complex30, Complex45, Complex52, Complex59, Complex61) no s'han obtingut resultats degut a errors en l'execució al clúster; en total s'obtenen 90 complexos vàlids.

Per a cada complex, el millor model en format .pdb es copia a la carpeta af3_pdbs_il7ra, que servirà com a input per als càlculs posteriors de Rosetta. Els resultats tabulats es guarden en format CSV a la carpeta af3_tables_il7ra.

In [1]:
import sys
from pathlib import Path
from prepare_proteins import sequenceModels

base = Path.cwd().parent  # La meva base és la carpeta "projects"
sys.path.append(str(base))

fasta_file = base / "Inputs" / "input_af3_il7ra.fasta"
af_folder = base / "Scripts" / "af3_jobs_il7ra"

# Carpeta on es guardaràn els fitxers .csv amb els resultats de ranking_scores
csv_folder = Path("./af3_tables_il7ra")
csv_folder.mkdir(parents=True, exist_ok=True)

# Carpeta on es guardaran les estructures PDB
pdb_folder = Path("./af3_pdbs_il7ra")
pdb_folder.mkdir(parents=True, exist_ok=True)

sequence_models = sequenceModels(str(fasta_file))

df_all_scores, df_selected_scores, copied_paths = sequence_models.collectAlphaFold3Results(
    af_folder=str(af_folder),
    return_selected=True,
    output_folder=str(pdb_folder),
    overwrite=True
)

print("\n=== TOTS ELS RESULTATS ===")
print(df_all_scores.head())

print("\n=== MILLOR PER MODEL ===")
print(df_selected_scores.head())

print("\nDimensions:")
print("scores_df:", df_all_scores.shape)
print("best_df:", df_selected_scores.shape)

print("\n=== PDBs GENERATS ===")
for model, files in copied_paths.items():
    print(model, "->", files)

/Users/bertaguiu/projects/prepare_proteins/prepare_proteins/_prepare_sequences.py:323: UserWarning: Sequences contain non-standard amino acid codes — Complex1: :, Complex2: :, Complex3: :, Complex4: :, Complex5: :, Complex6: :, Complex7: :, Complex8: :, Complex9: :, Complex10: :, Complex11: :, Complex12: :, Complex13: :, Complex14: :, Complex15: :, Complex16: :, Complex17: :, Complex18: :, Complex19: :, Complex20: :, Complex21: :, Complex22: :, Complex23: :, Complex24: :, Complex25: :, Complex26: :, Complex27: :, Complex28: :, Complex29: :, Complex30: :, Complex31: :, Complex32: :, Complex33: :, Complex34: :, Complex35: :, Complex36: :, Complex37: :, Complex38: :, Complex39: :, Complex40: :, Complex41: :, Complex42: :, Complex43: :, Complex44: :, Complex45: :, Complex46: :, Complex47: :, Complex48: :, Complex49: :, Complex50: :, Complex51: :, Complex52: :, Complex53: :, Complex54: :, Complex55: :, Complex56: :, Complex57: :, Complex58: :, Complex59: :, Complex60: :, Complex61: :, Compl


=== TOTS ELS RESULTATS ===
                      ranking_score job_seed_dir job_seed job_seed_instance  \
model    seed sample                                                          
Complex1 1    0            1.054455         None     None              None   
              1            1.061148         None     None              None   
              2            1.054278         None     None              None   
              3            1.056740         None     None              None   
              4            1.057111         None     None              None   

                     summary_chain_iptm       summary_chain_pair_iptm  \
model    seed sample                                                    
Complex1 1    0            [0.91, 0.91]  [[0.46, 0.91], [0.91, 0.84]]   
              1            [0.91, 0.91]  [[0.46, 0.91], [0.91, 0.83]]   
              2            [0.91, 0.91]  [[0.46, 0.91], [0.91, 0.83]]   
              3            [0.91, 0.91]  [[0.46, 0.91

In [2]:
df_all_scores.to_csv(csv_folder / "af3_scores_all.csv")
df_selected_scores.to_csv(csv_folder / "af3_scores_best.csv")

best_sorted = df_selected_scores.sort_values("ranking_score", ascending=False)
best_sorted.to_csv(csv_folder / "af3_scores_best_sorted.csv")

print("\n=== MILLORS MODELS ORDENATS PER ranking_score ===")
print(best_sorted)

print("\nCSV guardats a:", csv_folder)


=== MILLORS MODELS ORDENATS PER ranking_score ===
                       ranking_score job_seed_dir job_seed job_seed_instance  \
model     seed sample                                                          
Complex94 1    2            1.099656         None     None              None   
Complex26 1    1            1.097222         None     None              None   
Complex42 1    2            1.097032         None     None              None   
Complex41 1    2            1.095442         None     None              None   
Complex75 1    1            1.095049         None     None              None   
...                              ...          ...      ...               ...   
Complex67 1    3            0.810631         None     None              None   
Complex13 1    0            0.792535         None     None              None   
Complex24 1    2            0.703454         None     None              None   
Complex51 1    4            0.694233         None     None           